# 04 Business EDA

Muc tieu cua notebook nay la sinh bang tong hop va chart cho 4 theme EDA. Moi insight nen duoc viet theo khung What -> So what -> Implication -> Action.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'dataset').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from the current working directory.')

PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.competition_workflow import (
    build_eda_aggregate_tables,
    build_reviews_facts,
    ensure_stage_directories,
    export_eda_aggregates,
    load_tables,
    )

dirs = ensure_stage_directories(PROJECT_ROOT)
sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', None)

In [2]:
tables = load_tables(PROJECT_ROOT / 'dataset')
aggregates = build_eda_aggregate_tables(PROJECT_ROOT)
reviews_facts = build_reviews_facts(tables)
sorted(aggregates.keys())

['theme1_category_quarterly_revenue',
 'theme1_monthly_revenue',
 'theme1_revenue_seasonality',
 'theme1_segment_net_revenue',
 'theme2_bounce_by_source',
 'theme2_device_mix',
 'theme2_region_revenue',
 'theme2_source_sessions_orders',
 'theme3_delay_vs_rating',
 'theme3_rating_by_category',
 'theme3_return_rate_by_size_category',
 'theme3_return_reasons',
 'theme4_fill_rate',
 'theme4_lost_revenue',
 'theme4_sell_through',
 'theme4_stockout_days']

In [3]:
def save_line_chart(data, x, y, title, output_path):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(data[x], data[y], linewidth=2.5, color='#1f4e79')
    peak_row = data.loc[data[y].idxmax()]
    ax.scatter([peak_row[x]], [peak_row[y]], color='#c0392b', s=80)
    ax.annotate(f"Peak: {peak_row[y]:,.0f}", (peak_row[x], peak_row[y]), textcoords='offset points', xytext=(10, 10))
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    plt.xticks(rotation=45)
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

def save_bar_chart(data, x, y, title, output_path, color='#1f4e79'):
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=data, x=x, y=y, ax=ax, color=color)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

def save_heatmap(data, index, columns, values, title, output_path):
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot = data.pivot(index=index, columns=columns, values=values)
    sns.heatmap(pivot, cmap='Blues', ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

def save_scatter(data, x, y, title, output_path, hue=None, size=None):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.scatterplot(data=data, x=x, y=y, hue=hue, size=size, ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

def save_box_plot(data, x, y, title, output_path):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=data, x=x, y=y, ax=ax, color='#7eb0d5')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

In [4]:
artifact_paths = export_eda_aggregates(PROJECT_ROOT)
chart_dir = dirs['charts']

theme1_monthly = aggregates['theme1_monthly_revenue'].copy()
theme1_monthly['period'] = pd.to_datetime(theme1_monthly[['year', 'month']].assign(day=1))
save_line_chart(theme1_monthly, 'period', 'Revenue', 'Theme 1 - Monthly Revenue', chart_dir / 'theme1_monthly_revenue.png')
save_heatmap(aggregates['theme1_revenue_seasonality'], 'year', 'month', 'Revenue', 'Theme 1 - Revenue Seasonality', chart_dir / 'theme1_revenue_seasonality.png')
save_bar_chart(aggregates['theme1_category_quarterly_revenue'], 'quarter', 'line_revenue', 'Theme 1 - Category Contribution by Quarter', chart_dir / 'theme1_category_contribution.png')
save_bar_chart(aggregates['theme1_segment_net_revenue'].head(10), 'segment', 'net_revenue', 'Theme 1 - Top Segments by Net Revenue', chart_dir / 'theme1_segment_net_revenue.png')

save_bar_chart(aggregates['theme2_bounce_by_source'], 'traffic_source', 'avg_bounce_rate', 'Theme 2 - Bounce Rate by Traffic Source', chart_dir / 'theme2_bounce_rate_by_source.png')
save_scatter(aggregates['theme2_source_sessions_orders'], 'sessions', 'order_count', 'Theme 2 - Sessions vs Orders', chart_dir / 'theme2_sessions_vs_orders.png', hue='traffic_source', size='revenue')
save_bar_chart(aggregates['theme2_device_mix'], 'device_type', 'order_count', 'Theme 2 - Orders by Device', chart_dir / 'theme2_orders_by_device.png')
save_bar_chart(aggregates['theme2_region_revenue'], 'region', 'revenue', 'Theme 2 - Revenue by Region', chart_dir / 'theme2_revenue_by_region.png')

save_heatmap(aggregates['theme3_return_rate_by_size_category'], 'size', 'category', 'return_rate', 'Theme 3 - Return Rate by Size and Category', chart_dir / 'theme3_return_rate_heatmap.png')
save_bar_chart(aggregates['theme3_return_reasons'], 'return_reason', 'return_count', 'Theme 3 - Return Reasons', chart_dir / 'theme3_return_reasons.png')
save_scatter(aggregates['theme3_delay_vs_rating'], 'delay_days', 'avg_rating', 'Theme 3 - Shipping Delay vs Rating', chart_dir / 'theme3_delay_vs_rating.png')
save_box_plot(reviews_facts.dropna(subset=['category', 'rating']), 'category', 'rating', 'Theme 3 - Review Rating by Category', chart_dir / 'theme3_rating_by_category.png')

save_bar_chart(aggregates['theme4_stockout_days'], 'category', 'stockout_days', 'Theme 4 - Stockout Days by Category', chart_dir / 'theme4_stockout_days.png')
save_heatmap(aggregates['theme4_fill_rate'], 'category', 'month_key', 'fill_rate', 'Theme 4 - Fill Rate by Category and Month', chart_dir / 'theme4_fill_rate_heatmap.png')
save_bar_chart(aggregates['theme4_sell_through'], 'category', 'sell_through_rate', 'Theme 4 - Sell Through Rate by Category', chart_dir / 'theme4_sell_through.png')
save_bar_chart(aggregates['theme4_lost_revenue'], 'category', 'estimated_lost_revenue', 'Theme 4 - Estimated Lost Revenue', chart_dir / 'theme4_lost_revenue.png')

pd.DataFrame({
    'artifact': list(artifact_paths.keys()),
    'path': [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths.values()],
}).sort_values('artifact')

,artifact,path
2,theme1_category_quarterly_revenue,artifacts\eda\theme1_category_quarterly_revenu...
0,theme1_monthly_revenue,artifacts\eda\theme1_monthly_revenue.csv
1,theme1_revenue_seasonality,artifacts\eda\theme1_revenue_seasonality.csv
3,theme1_segment_net_revenue,artifacts\eda\theme1_segment_net_revenue.csv
4,theme2_bounce_by_source,artifacts\eda\theme2_bounce_by_source.csv
6,theme2_device_mix,artifacts\eda\theme2_device_mix.csv
7,theme2_region_revenue,artifacts\eda\theme2_region_revenue.csv
5,theme2_source_sessions_orders,artifacts\eda\theme2_source_sessions_orders.csv
10,theme3_delay_vs_rating,artifacts\eda\theme3_delay_vs_rating.csv
11,theme3_rating_by_category,artifacts\eda\theme3_rating_by_category.csv


## Insight template

### Theme 1 - Revenue and Demand
**What:** ...  
**So what:** ...  
**Implication:** ...  
**Action:** ...

### Theme 2 - Customer and Channel
**What:** ...  
**So what:** ...  
**Implication:** ...  
**Action:** ...

### Theme 3 - Returns and Experience
**What:** ...  
**So what:** ...  
**Implication:** ...  
**Action:** ...

### Theme 4 - Inventory and Operations
**What:** ...  
**So what:** ...  
**Implication:** ...  
**Action:** ...